

#### ORPO (Optimized Retrieval and Pretraining Objective) is a technique used in large language models (LLMs) to enhance their retrieval and reasoning capabilities. It is designed to improve how LLMs retrieve relevant information from external sources and integrate that knowledge into responses. ORPO typically combines retrieval-augmented generation (RAG) techniques with specialized pretraining objectives to refine the model’s ability to generate context-aware and accurate outputs.



In [1]:
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

!pip install --no-deps "xformers<0.0.26" trl peft accelerate bitsandbytes


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-2rg1cqqp/unsloth_5bca25a393ea4949a0a39f3e0c3d7dfb
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-2rg1cqqp/unsloth_5bca25a393ea4949a0a39f3e0c3d7dfb
  Resolved https://github.com/unslothai/unsloth.git to commit 5df2a0ce2a63a8b206c2e857bb44f4f9247610f5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.7/123.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.7/123.7 kB 10.7 MB/s eta 0:00:0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 MB 5.5 MB/s eta 0:00:00


In [14]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.3.14: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

## Data Prep
You need at least 3 columns:

- Instruction
- Accepted
- Rejected
#### For example:
- Instruction: "What is 2+2?"
- Accepted: "The answer is 4"
- Rejected: "The answer is 5"

The goal of ORPO is to penalize the "rejected" samples, and increase the likelihood of "accepted" samples. recipe-research essentially used Mistral to generate the "rejected" responses, and used GPT-4 to generated the "accepted" responses.

In [10]:
alpaca_prompt =  """ Below is an instruction that describes a task, paired with an input that provides further context. write a responce that approxiately completes the request.

### Instruction:
{}

### Input:
{}


### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # must add eos token
def format_prompt(sample):
    instruction = sample["instruction"]  # Extract the instruction from the sample
    input = sample["input"]              # Extract the input from the sample
    accepted = sample["accepted"]        # Extract the accepted response from the sample
    rejected = sample["rejected"]        # Extract the rejected response from the sample

    # ORPOTrainer expects prompt/chosen/rejected keys
    sample["prompt"] = alpaca_prompt.format(instruction, input, "")  # Format the prompt using the template
    sample["chosen"] = accepted + EOS_TOKEN  # Append EOS token to the accepted response
    sample["rejected"] = rejected + EOS_TOKEN  # Append EOS token to the rejected response

    return sample  # Return the modified sample

pass



In [8]:
from datasets import load_dataset

dataset = load_dataset("reciperesearch/dolphin-sft-v0.1-preference")["train"]

In [11]:
# This function is then applied to each sample in the dataset using dataset.map,
# which processes each sample through the format_prompt function and returns the formatted dataset.
dataset = dataset.map(format_prompt,)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

In [12]:
# Enable reward modelling stats
from unsloth import PatchDPOTrainer
PatchDPOTrainer()

In [15]:
from trl import ORPOConfig, ORPOTrainer

orpo_trainer = ORPOTrainer(
    model = model,
    train_dataset = dataset,
    tokenizer = tokenizer,
    args = ORPOConfig(
        max_length = max_seq_length,
        max_prompt_length = max_seq_length//2,
        max_completion_length = max_seq_length//2,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        beta = 0.1,
        logging_steps = 1,
        optim = "adamw_8bit",
        lr_scheduler_type = "linear",
        max_steps = 30, # Change to num_train_epochs = 1 for full training runs
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        output_dir = "outputs",
    ),
)

/content/unsloth_compiled_cache/UnslothORPOTrainer.py:556: UserWarning: When using DPODataCollatorWithPadding, you should set `remove_unused_columns=False` in your TrainingArguments we have set it for you, but you should do it yourself in the future.
  warnings.warn(


Map (num_proc=2):   0%|          | 0/16000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/16000 [00:00<?, ? examples/s]

In [32]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import Accelerator
from trl import ORPOConfig, ORPOTrainer

# Initialize accelerator
accelerator = Accelerator()

# Load your model with device_map="auto"
model = AutoModelForCausalLM.from_pretrained(
    "unsloth/llama-3-8b-bnb-4bit",  # Make sure this variable is defined
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

# Then initialize the ORPO trainer
orpo_trainer = ORPOTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=ORPOConfig(
        max_length=max_seq_length,
        max_prompt_length=max_seq_length//2,
        max_completion_length=max_seq_length//2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        beta=0.1,
        logging_steps=1,
        optim="adamw_8bit",
        lr_scheduler_type="linear",
        max_steps=30,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        output_dir="outputs",
    ),
)

# Now train should work
orpo_trainer.train()

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

## Inference


In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonnaci sequence.", # instruction
        "1, 1, 2, 3, 5, 8", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)

tokenizer.batch_decode(outputs)